In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [6]:
df = pd.read_csv("../data/df_dropped_rows.csv")
df = df[['Party', 'speech_text']].dropna()
label_map = {"Republican": 0, "Democrat": 1}
df["labels"] = df["Party"].map(label_map)
df = df.dropna()


In [4]:

# input texts and labeling
X_train, X_test, y_train, y_test = train_test_split(
    df["speech_text"],
    df["labels"],
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)
#deals w long speeches
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True
)
# learn the vocab from trianing speeches and convert every speech into tf-idf vector
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

#uses gradient descent 
clf = LogisticRegression(
    max_iter=3000,
    n_jobs=-1
)
clf.fit(X_train_vec, y_train)
y_pred = clf.predict(X_test_vec)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Republican", "Democrat"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

  Republican       0.70      0.71      0.70      2268
    Democrat       0.72      0.71      0.72      2380

    accuracy                           0.71      4648
   macro avg       0.71      0.71      0.71      4648
weighted avg       0.71      0.71      0.71      4648


Confusion Matrix:
[[1603  665]
 [ 680 1700]]


In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]  

top_dem_idx = np.argsort(coefs)[::-1][:15]

print("Top Democrat-associated terms:\n")
for i in top_dem_idx:
    print(f"{feature_names[i]:30s}  coef={coefs[i]:.4f}")

top_rep_idx = np.argsort(coefs)[:15]

print("\nTop Republican associated terms:\n")
for i in top_rep_idx:
    print(f"{feature_names[i]:30s}  coef={coefs[i]:.4f}")


Top Democrat-associated terms:

republican                      coef=4.8558
cuts                            coef=3.9617
workers                         coef=2.6338
republicans                     coef=2.6289
cut                             coef=2.2980
republican colleagues           coef=2.1589
majority                        coef=2.0988
public                          coef=2.0653
health                          coef=2.0557
caucus                          coef=1.9322
investments                     coef=1.9255
communities                     coef=1.9134
families                        coef=1.9004
iraq                            coef=1.8102
children                        coef=1.7969

Top Republican associated terms:

spending                        coef=-4.3652
actually                        coef=-2.5858
trillion                        coef=-2.4914
democrat                        coef=-2.4830
reforms                         coef=-2.3303
china                           coef=-2.2649
tax